# Lab 5


Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.


2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows. 
    * Modify `__getitem__` implemented above to support slicing.
        

3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don’t forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N
    * M=N


6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [5]:
class Matrix:
    def __init__(self, *args):
        if len(args) == 2 and isinstance(args[0], int):
            self.data = [[0.0 for _ in range(args[1])] for _ in range(args[0])]
        elif len(args) == 1 and isinstance(args[0], list):
            if not all(len(row) == len(args[0][0]) for row in args[0]):
                raise ValueError("Inconsistent row lengths.")
            self.data = [list(row) for row in args[0]]

    def shape(self):
        return (len(self.data), len(self.data[0]))

    def to_list(self):
        return [list(row) for row in self.data]
    def copy_from(self, other):
        """Logic for 'Matrix assignment' M1 = M2 (value-wise)"""
        other_data = other.data if isinstance(other, Matrix) else other
        other_shape = (len(other_data), len(other_data[0]))
        if self.shape() != other_shape:
            raise ValueError("Dimension mismatch for assignment.")
        self.data = [list(row) for row in other_data]

    def transpose(self):
        n, m = self.shape()
        return Matrix([[self.data[j][i] for j in range(n)] for i in range(m)])

    def row(self, n):
        return Matrix([list(self.data[n])])

    def column(self, m):
        return Matrix([[row[m]] for row in self.data])

    def block(self, n0, n1, m0, m1):
        return Matrix([row[m0:m1] for row in self.data[n0:n1]])

    def __getitem__(self, key):
        if isinstance(key, tuple):
            r, c = key
            if isinstance(r, slice) or isinstance(c, slice):
                rows = self.data[r] if isinstance(r, slice) else [self.data[r]]
                return Matrix([row[c] if isinstance(c, slice) else [row[c]] for row in rows])
            return self.data[r][c]
        return self.data[key]
    def scalarmul(self, c):
        return Matrix([[val * c for val in row] for row in self.data])

    def add(self, N):
        if self.shape() != N.shape(): raise ValueError("Shapes must match for addition.")
        return Matrix([[self[i,j] + N[i,j] for j in range(self.shape()[1])] for i in range(self.shape()[0])])

    def sub(self, N):
        if self.shape() != N.shape(): raise ValueError("Shapes must match for subtraction.")
        return Matrix([[self[i,j] - N[i,j] for j in range(self.shape()[1])] for i in range(self.shape()[0])])

    def mat_mult(self, N):
        if self.shape()[1] != N.shape()[0]: raise ValueError("Incompatible dimensions for product.")
        res = Matrix(self.shape()[0], N.shape()[1])
        for i in range(self.shape()[0]):
            for j in range(N.shape()[1]):
                res.data[i][j] = sum(self[i, k] * N[k, j] for k in range(self.shape()[1]))
        return res

    def element_mult(self, N):
        if self.shape() != N.shape(): raise ValueError("Shapes must match for element-wise mult.")
        return Matrix([[self[i,j] * N[i,j] for j in range(self.shape()[1])] for i in range(self.shape()[0])])

    def equals(self, N):
        if not isinstance(N, Matrix) or self.shape() != N.shape(): return False
        return all(self[i,j] == N[i,j] for i in range(self.shape()[0]) for j in range(self.shape()[1]))
    def __add__(self, other): return self.add(other)
    def __sub__(self, other): return self.sub(other)
    def __mul__(self, other): 
        return self.mat_mult(other) if isinstance(other, Matrix) else self.scalarmul(other)
    def __rmul__(self, other): return self.scalarmul(other) # For 2 * M
    def __eq__(self, other): return self.equals(other)
    def __repr__(self): return str(self.data)

In [6]:
def constant(n, m, c):
    return Matrix([[float(c) for _ in range(m)] for _ in range(n)])

def zeros(n, m):
    return constant(n, m, 0)

def ones(n, m):
    return constant(n, m, 1)

def eye(n):
    m = zeros(n, n)
    for i in range(n): m.data[i][i] = 1.0
    return m

In [7]:
A = Matrix([[1, 2], [3, 4]])
B = Matrix([[5, 6], [7, 8]])
C = Matrix([[1, 0], [2, 1]])
I = eye(2)

print(f"A:\n{A}\nB:\n{B}\nC:\n{C}\n")
lhs = (A * B) * C
rhs = A * (B * C)
print(f"Associativity (AB)C == A(BC): {lhs == rhs}")

lhs = A * (B + C)
rhs = (A * B) + (A * C)
print(f"Distributivity A(B+C) == AB + AC: {lhs == rhs}")
print(f"Non-commutativity AB == BA: {A * B == B * A}")

print(f"Identity AI == A: {A * I == A}")

print(f"Scalar multiplication (2 * A):\n{2 * A}")

A:
[[1, 2], [3, 4]]
B:
[[5, 6], [7, 8]]
C:
[[1, 0], [2, 1]]

Associativity (AB)C == A(BC): True
Distributivity A(B+C) == AB + AC: True
Non-commutativity AB == BA: False
Identity AI == A: True
Scalar multiplication (2 * A):
[[2, 4], [6, 8]]
